# 🎨 Neural Style Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TUSHARTAMRAKAR/Neural-Style-Transfer/blob/main/notebook/nst_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-TUSHARTAMRAKAR-181717?logo=github)](https://github.com/TUSHARTAMRAKAR/Neural-Style-Transfer)

**Implementation of:** *A Neural Algorithm of Artistic Style* — Gatys, Ecker & Bethge (2015) · [arxiv:1508.06576](https://arxiv.org/abs/1508.06576)

---

### What you will learn
- How VGG-19 extracts content and style features at different depths
- What Gram matrices are and why they capture texture without position
- How L-BFGS optimization rewrites pixel values iteratively
- The exact weight balance needed for portrait vs landscape photos

### Before starting — enable GPU!
**Runtime → Change runtime type → T4 GPU → Save**

GPU: ~30 seconds per run · CPU: ~5-8 minutes per run

---
## Step 1 — Check GPU and environment

In [ ]:
import torch
import sys

print('=' * 50)
print(f'Python:  {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('\n✅ GPU ready — runs in ~30 seconds!')
else:
    print('\n⚠️  No GPU — runs in ~5-8 min. Go to Runtime → Change runtime type → T4 GPU')
print('=' * 50)

---
## Step 2 — Clone our NST engine from GitHub

In [ ]:
# Clone the production NST engine — same code powering the live web app
!git clone https://github.com/TUSHARTAMRAKAR/Neural-Style-Transfer.git /tmp/nst --quiet
sys.path.insert(0, '/tmp/nst/backend')

from nst_engine import run_style_transfer, get_style_presets, STYLE_PRESETS
print('✅ NST engine loaded!')
print(f'Available presets: {list(STYLE_PRESETS.keys())}')

---
## Step 3 — Get your images

**Option A:** Upload your own photos (recommended)

**Option B:** Use our preset artworks downloaded automatically

In [ ]:
# ── OPTION A: Upload your own images ─────────────────────────
# Run this cell to upload from your computer

from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

print('Upload your CONTENT image (your photo):')
uploaded = files.upload()
content_path = list(uploaded.keys())[0]
print(f'✅ Content: {content_path}')

print('\nUpload your STYLE image (artwork):')
uploaded = files.upload()
style_path = list(uploaded.keys())[0]
print(f'✅ Style: {style_path}')

In [ ]:
# ── OPTION B: Use preset artworks (skip if you ran Option A) ──
# Downloads style images automatically from GitHub
# Change PRESET below to: starry_night / the_scream / kandinsky / mosaic / wave / udnie

import urllib.request, os

PRESET = 'starry_night'   # ← change this!

GITHUB_RAW = 'https://raw.githubusercontent.com/TUSHARTAMRAKAR/Neural-Style-Transfer/main/docs/style_images'
style_path = f'/tmp/{PRESET}.jpg'

if not os.path.exists(style_path):
    print(f'Downloading {PRESET}.jpg...')
    urllib.request.urlretrieve(f'{GITHUB_RAW}/{PRESET}.jpg', style_path)
    print(f'✅ Downloaded: {os.path.getsize(style_path)//1024}KB')
else:
    print(f'✅ {PRESET}.jpg already cached')

# For content: use a sample image or set content_path manually
# content_path = 'your_photo.jpg'  # set this to your uploaded file
print(f'\nPreset info: {STYLE_PRESETS[PRESET]["name"]} by {STYLE_PRESETS[PRESET]["artist"]}')
print(f'Description: {STYLE_PRESETS[PRESET]["description"]}')

In [ ]:
# Preview both images side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Image.open(content_path))
axes[0].set_title('Content image (your photo)', fontsize=13, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(Image.open(style_path))
axes[1].set_title('Style image (the artwork)', fontsize=13, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

---
## Step 4 — Visualize what VGG-19 sees

Before running NST, let's look inside the network. VGG-19 has 19 layers — each layer sees the image differently:
- **Early layers** (relu1_1): edges, colors, simple textures
- **Middle layers** (relu3_1): shapes, patterns, object parts  
- **Deep layers** (relu4_2): objects, faces, high-level structure ← we use this for content

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load VGG-19 feature extractor
print('Loading VGG-19...')
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(DEVICE).eval()
print(f'✅ VGG-19 loaded on {DEVICE}')

# Prepare image tensor
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
img = transform(Image.open(content_path).convert('RGB')).unsqueeze(0).to(DEVICE)

# Register hooks to capture feature maps at 3 layers
activations = {}
vgg[1].register_forward_hook(lambda m,i,o: activations.update({'relu1_1': o.detach()}))
vgg[10].register_forward_hook(lambda m,i,o: activations.update({'relu3_1': o.detach()}))
vgg[22].register_forward_hook(lambda m,i,o: activations.update({'relu4_2': o.detach()}))

with torch.no_grad():
    vgg(img)

# Plot 8 channels from each layer
fig, axes = plt.subplots(3, 8, figsize=(18, 8))
layer_info = [
    ('relu1_1', 'Layer relu1_1 — detects edges and colors'),
    ('relu3_1', 'Layer relu3_1 — detects shapes and patterns'),
    ('relu4_2', 'Layer relu4_2 — detects objects (CONTENT LAYER)'),
]

for row, (layer, title) in enumerate(layer_info):
    feat = activations[layer][0].cpu().numpy()
    axes[row, 0].set_ylabel(title, fontsize=8, rotation=90, labelpad=5, va='center')
    for col in range(8):
        if col < feat.shape[0]:
            axes[row, col].imshow(feat[col], cmap='viridis')
        axes[row, col].axis('off')

plt.suptitle('VGG-19 Feature Maps — What the network sees at different depths', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('Notice: early layers preserve image structure, deep layers show abstract shapes')

---
## Step 5 — Visualize the Gram matrix

The Gram matrix is **the secret of style transfer**.

Instead of comparing feature maps directly (which would just copy the layout), we compare **channel correlations**:

```
Gram[i, j] = dot product of channel i and channel j
           = 'How much do these two texture detectors fire together?'
```

This captures texture statistics **independent of position** — same Gram matrix = same artistic style, regardless of what's in the image.

In [ ]:
def gram_matrix(features):
    b, c, h, w = features.size()
    f = features.view(b * c, h * w)      # flatten spatial dims
    gram = torch.mm(f, f.t())            # channel × channel correlations
    return gram.div(b * c * h * w)       # normalize by size

# Get style image features
style_img = transform(Image.open(style_path).convert('RGB')).unsqueeze(0).to(DEVICE)
style_acts = {}
vgg[1].register_forward_hook(lambda m,i,o: style_acts.update({'relu1_1': o.detach()}))

with torch.no_grad():
    vgg(style_img)

feat = style_acts['relu1_1']
gram = gram_matrix(feat).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(Image.open(style_path))
axes[0].set_title('Style image', fontweight='bold', fontsize=12)
axes[0].axis('off')

im1 = axes[1].imshow(gram[:32, :32], cmap='plasma')
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Gram matrix (32×32 subset)\nHigh value = channels co-activate', fontweight='bold', fontsize=10)

im2 = axes[2].imshow(np.log(np.abs(gram[:64, :64]) + 1), cmap='inferno')
plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Log scale — shows all correlations\nThis encodes the full artistic texture', fontweight='bold', fontsize=10)

plt.suptitle('The Gram matrix captures texture statistics — the entire style in one 64×64 matrix', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f'Gram matrix shape: {gram.shape} (channels × channels)')
print(f'Each cell = correlation between two texture detectors in VGG-19')

---
## Step 6 — Choose your settings

The right settings depend on what you are stylizing:

| Subject | Content weight | Style weight | Steps |
|---------|---------------|-------------|-------|
| Portrait / face | 15,000 | 80,000,000 | 300 |
| Landscape / scene | 5,000 | 400,000,000 | 400 |
| Max artistic effect | 1,000 | 900,000,000 | 400 |

In [ ]:
# ── Choose your subject mode ──────────────────────────────────
# Options: 'portrait'  'landscape'  'max_style'
SUBJECT_MODE = 'portrait'

configs = {
    'portrait':  {'content_weight': 15000,  'style_weight': 80_000_000,  'num_steps': 300},
    'landscape': {'content_weight': 5000,   'style_weight': 400_000_000, 'num_steps': 400},
    'max_style': {'content_weight': 1000,   'style_weight': 900_000_000, 'num_steps': 400},
}

cfg = configs[SUBJECT_MODE]
print(f'Subject mode:     {SUBJECT_MODE}')
print(f'Content weight:   {cfg["content_weight"]:,}')
print(f'Style weight:     {cfg["style_weight"]:,}')
print(f'Steps:            {cfg["num_steps"]}')
print(f'Style/content ratio: {cfg["style_weight"]//cfg["content_weight"]:,}x')
print(f'\nEstimated time: {"~30 sec" if torch.cuda.is_available() else "~5-8 min"}')

---
## Step 7 — Run Neural Style Transfer 🚀

In [ ]:
import time

OUTPUT_PATH = 'stylized_result.png'
steps_log   = []

def log_progress(step, total, content_loss, style_loss, total_loss):
    steps_log.append({
        'step': step,
        'content_loss': content_loss,
        'style_loss': style_loss,
        'total_loss': total_loss,
    })
    bar = '█' * int(30 * step / total) + '░' * (30 - int(30 * step / total))
    print(f'[{bar}] {step}/{total} | Loss: {total_loss:.2e}', end='\r')

print(f'Starting NST on {DEVICE}...')
print(f'Steps: {cfg["num_steps"]} · Style weight: {cfg["style_weight"]:,}\n')

start = time.time()
result_path = run_style_transfer(
    content_path=content_path,
    style_path=style_path,
    output_path=OUTPUT_PATH,
    num_steps=cfg['num_steps'],
    content_weight=cfg['content_weight'],
    style_weight=cfg['style_weight'],
    progress_callback=log_progress,
)
elapsed = time.time() - start

print(f'\n\n✅ Done in {elapsed:.0f}s! Saved to: {result_path}')

---
## Step 8 — View results and loss curve

In [ ]:
# Show before / style / after comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(Image.open(content_path))
axes[0].set_title('Content image', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(Image.open(style_path))
axes[1].set_title('Style image', fontsize=13, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(Image.open(result_path))
axes[2].set_title('Stylized result ✨', fontsize=13, fontweight='bold', color='#6366f1')
axes[2].axis('off')

plt.suptitle(f'Neural Style Transfer Result — {SUBJECT_MODE} mode · {elapsed:.0f}s on {DEVICE}', fontsize=12)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved comparison.png')

In [ ]:
# Plot the loss curve — shows optimization converging
if steps_log:
    steps    = [s['step'] for s in steps_log]
    c_losses = [s['content_loss'] for s in steps_log]
    s_losses = [s['style_loss'] for s in steps_log]
    t_losses = [s['total_loss'] for s in steps_log]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(steps, t_losses, color='#6366f1', linewidth=2)
    axes[0].set_title('Total loss', fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].grid(alpha=0.3)

    axes[1].plot(steps, c_losses, color='#06b6d4', linewidth=2)
    axes[1].set_title('Content loss', fontweight='bold')
    axes[1].set_xlabel('Step')
    axes[1].grid(alpha=0.3)

    axes[2].plot(steps, s_losses, color='#a855f7', linewidth=2)
    axes[2].set_title('Style loss', fontweight='bold')
    axes[2].set_xlabel('Step')
    axes[2].grid(alpha=0.3)

    plt.suptitle('Loss curves — all three should decrease steadily (optimization working)', fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f'Final total loss: {t_losses[-1]:.4e}')
    print(f'Loss reduced by:  {(1 - t_losses[-1]/t_losses[0])*100:.1f}%')

In [ ]:
# Download your result
from google.colab import files
files.download(result_path)
files.download('comparison.png')
print('Downloading stylized result and comparison...')

---
## Key takeaways

| Concept | What it does |
|---|---|
| **VGG-19** | Pretrained CNN — 19 layers of increasingly abstract feature detection |
| **Content loss** | MSE between deep feature maps (conv4_2) — preserves structure |
| **Style loss** | MSE between Gram matrices across 5 layers — captures texture |
| **L-BFGS** | Quasi-Newton optimizer — converges in 300-400 steps vs 2000+ for Adam |
| **Gram matrix** | Channel correlations — position-independent texture statistics |
| **Style weight** | Must be 1e8–1e9 to visibly overpower photo structure |

---

### Further resources
- Original paper: [Gatys et al., 2015](https://arxiv.org/abs/1508.06576)
- VGG architecture: [Simonyan & Zisserman, 2014](https://arxiv.org/abs/1409.1556)
- Fast NST (1000x faster): [Johnson et al., 2016](https://arxiv.org/abs/1603.08155)
- Live web app: [huggingface.co/spaces/Tusharz/Neural-Style-Transfer](https://huggingface.co/spaces/Tusharz/Neural-Style-Transfer)
- Full source code: [github.com/TUSHARTAMRAKAR/Neural-Style-Transfer](https://github.com/TUSHARTAMRAKAR/Neural-Style-Transfer)

---
*Made with love by [Tushar Tamrakar](https://github.com/TUSHARTAMRAKAR)*